# RNA-seq count-table notebook (airway GSE52778 subset)

Learning notebook. Same analysis as `src/analyze.py`. Default data: real public recount2 counts for GSE52778/SRP033351 (trimmed). Not a production pipeline.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[0] / 'src') if Path.cwd().name == 'notebooks' else 'src')
from analyze import log_cpm, bh_fdr, DATA
import pandas as pd
from scipy import stats

counts = pd.read_csv(DATA / 'airway_dex_counts.csv', index_col=0)
meta = pd.read_csv(DATA / 'airway_sample_metadata.csv').set_index('sample')
print(counts.shape)
print(meta['condition'].value_counts())
lcpm = log_cpm(counts[meta.index])
ctrl = lcpm.loc[:, meta['condition']=='control']
trt = lcpm.loc[:, meta['condition']=='treated']
log2fc = trt.mean(axis=1) - ctrl.mean(axis=1)
p = [stats.ttest_ind(trt.loc[g], ctrl.loc[g], equal_var=False).pvalue for g in lcpm.index]
out = pd.DataFrame({'log2fc': log2fc, 'pvalue': p})
out['fdr'] = bh_fdr(out['pvalue'].to_numpy())
print(out.sort_values('fdr').head())
